In [1]:
from astropy.io import fits
import numpy as np
from astropy.table import Table
import pandas as pd

In [14]:
hdul = fits.open("CIV-Absorbers-dr1-v1.0.fits") # lya quasar catalog
catalog_absorbers = hdul[1].data
catalog_absorbers = catalog_absorbers['Z_ABS']
catalog_IDs = hdul[2].data['TARGETID'].astype(str)  # Ensure TARGETID is string type

hdul = fits.open("CIV-Absorbers-parent-QSO-dr1-v1.0.fits")
parent_IDs = hdul[1].data['TARGETID'].astype(str)

df = pd.read_csv("CIV_catalog-validationtest1.csv", dtype={0:str})
detected_IDs = df.iloc[:,0]
detected_absorbers = df.iloc[:,[1,2,3,4,5,6,7]]

In [3]:
# print(catalog_IDs)
# print(detected_IDs)
# print(catalog_absorbers)
# print(detected_absorbers)
print(type(detected_IDs[1]))
print(detected_IDs[1])

<class 'str'>
39627797563442938


In [9]:
dv = 100 # km/s, velocity difference threshold
# Convert dv to redshift difference
c = 299792.458  # speed of light in km/s
z_trh = dv / c

TrueP = 0
TrueP2 = 0
FalseP = 0
FalseN = 0

# ...existing code...
catalog_absorbers = np.array(catalog_absorbers, dtype=float)
detected_absorbers = np.array(detected_absorbers, dtype=float)
# ...existing code...
# Create a dictionary mapping TARGETID to Z_ABS
catalog_dict = {tid: zabs for tid, zabs in zip(catalog_IDs, catalog_absorbers)}
detected_dict = {tid: zabs for tid, zabs in zip(detected_IDs, detected_absorbers)}

for i in range(len(detected_IDs)):  #loops over all detected IDs
    tid = detected_IDs[i]           #sets working ID
    if tid in catalog_dict:         #proceeds if the ID matches to catalog
        catalog_zabs = np.array(catalog_dict[tid],ndmin=1)        #Create arrays for both sets of absorbers
        detected_zabs = np.array(detected_dict[tid],ndmin=1)
        dz = np.zeros(np.size(catalog_zabs))      # Initialize dz array
        bool_match = np.zeros(np.size(catalog_zabs), dtype=bool)
        for j in range(len(detected_zabs)):     #Searching thru our detected absorbers
            current_z = detected_zabs[j]
            for k in range(np.size(catalog_zabs)):
                dz[k] = catalog_zabs[k] - current_z
                bool_match[k] = np.abs(dz[k]) < z_trh
            if sum(bool_match) > 0:             #Absorbers match, true positive
                TrueP += 1
            elif sum(bool_match) == 0:          #Absorbers do not match, false positive, we found it but they didn't
                FalseP += 1
        for j in range(len(catalog_zabs)):      #Searching thru the catalog's absorbers
            current_z = catalog_zabs[j]
            dz = detected_zabs - current_z
            bool_match = np.abs(dz) < z_trh
            if sum(bool_match) > 0:             #Absorbers match, true positive (redundant)   
                TrueP2 += 1
            elif sum(bool_match) == 0:          #Absorbers do not match, false negative, they found it but we didn't
                FalseN += 1
    else:
        detected_zabs = np.array(detected_dict[tid],ndmin=1)
        detected_zabs = detected_zabs[~np.isnan(detected_zabs)]
        FalseP += len(detected_zabs)

print("True Positive:", TrueP)
print("False Positive:", FalseP)
print("False Negative:", FalseN)
print(" ")

purity = TrueP / (TrueP + FalseP)
completeness = TrueP / (TrueP + FalseN)
print("Purity: ", purity*100, '%')
print("Completeness:", completeness*100, '%')

True Positive: 192
False Positive: 2442
False Negative: 52
 
Purity:  7.289293849658314 %
Completeness: 78.68852459016394 %


In [10]:
print(TrueP+FalseP)
print(TrueP+FalseN)

2634
244


In [15]:
#Detected IDs not in catalog
detected_only_ids = [tid for tid in detected_dict if tid not in parent_IDs]
print(detected_only_ids)
print(len(detected_only_ids), "detected IDs not in catalog")

[]
0 detected IDs not in catalog


In [16]:
#Common IDs
common_ids = list(set(catalog_dict.keys()) & set(detected_dict.keys()))
print(common_ids)
print(len(common_ids), "common IDs between catalog absorbers and detected absorbers")

['39627821718439126', '39627888059747196', '39627936193583886', '39628475925007582', '39628097116441536', '39628073473151314', '39628507218707020', '39627809634648198', '39627869978104522', '39628020020938754', '39627984188999667', '39627966228989009', '39627942229182480', '39627930267029409', '39627888114273647', '39627972155542713', '39627954212308939', '39627996142767463', '39627918200013488', '39627990149107475', '39627906120421111', '39627803603242846', '39627942166269587', '39627954275226708', '39627960226939328', '39627966224797661', '39627948222844068', '39627863959279612', '39627876013708256', '39627888114271917', '39628031936955796', '39628507197739687', '39627833798036300', '39628067588542317', '39628502021965410', '39627960201774879', '39627960239523587', '39628502013576547', '39627918183237640', '39628507193544641', '39628102992659479', '39628055748019231', '39628079387117992', '39628043819420748', '39628091223443428', '39628008012646662', '39627900135148378', '39627990111